# Проект 1. Часть 1. Приток: сколько клиентов приходит по месяцам

**Вопрос этой части:** как менялся приток клиентов за всю историю и одинаково ли он распределён по каналам?
Это отправная точка: дальше от неё строятся гипотеза и факторная декомпозиция.

**Что считаем (три разных события, их нельзя смешивать):**

| Метрика | Одна строка источника | Событие | Месяц берём по |
|---|---|---|---|
| Регистрации | клиент (`clients`) | пришёл в экосистему | `registered_at` |
| Новые подписки | оформление (`subscriptions`) | оформил подписку | `started_at` |
| Новые платящие | клиент | первое успешное списание в жизни | минимум `paid_at` при `status = 'success'` |

**Зерно итоговой таблицы:** месяц × канал привлечения. Канал у подписки и платежа — канал владельца подписки.
Клиентов без канала не выбрасываем, а держим отдельной строкой «без канала».
Сутки и месяцы считаем по UTC.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from sqlalchemy import text

from prime.config import get_engine

engine = get_engine()


def q(sql: str, **params) -> pd.DataFrame:
    """Выполнить запрос и вернуть DataFrame. Соединение закрывается сразу."""
    try:
        with engine.connect() as conn:
            conn.execute(text("set time zone 'UTC'"))   # сутки и месяцы — по UTC
            return pd.read_sql(text(sql), conn, params=params)
    finally:
        engine.dispose()


NO_CHANNEL = "без канала"
print("подключились как:", q("select current_user as login").iloc[0, 0])

## 1. Границы данных

Прежде чем считать, смотрим, за какой период вообще есть каждое событие. Это определяет, каким месяцам верить.

In [ ]:
bounds = q("""
    select 'регистрации (clients.registered_at)'      as event, min(registered_at) as first_at, max(registered_at) as last_at, count(*) as n
    from prime.clients
    union all
    select 'подписки (subscriptions.started_at)',     min(started_at), max(started_at), count(*)
    from prime.subscriptions
    union all
    select 'платежи (payments.paid_at, все статусы)', min(paid_at), max(paid_at), count(*)
    from prime.payments
""")
bounds

**Что смотреть в таблице выше.**
Если первые подписки или платежи начинаются позже первых регистраций (регистрации идут с января 2024), то «приток» подписок и платящих в ранних месяцах считается по неполной истории.
Последний месяц данных, скорее всего, неполный: сравнивать его с остальными в абсолютных числах нельзя.

## 2. Регистрации по месяцам и каналам

In [ ]:
reg = q("""
    select date_trunc('month', registered_at)::date          as month,
           coalesce(acquisition_channel, :no_channel)        as channel,
           count(*)                                          as registered
    from prime.clients
    group by 1, 2
    order by 1, 2
""", no_channel=NO_CHANNEL)

total_clients = q("select count(*) as n from prime.clients").iloc[0, 0]

# Контроль до всего остального: сумма по месяцам и каналам обязана равняться числу клиентов.
assert reg["registered"].sum() == total_clients, "регистрации по группам не сходятся с общим числом клиентов"
print(f"клиентов всего: {total_clients}, месяцев: {reg['month'].nunique()}, каналов: {reg['channel'].nunique()}")
reg.head(8)

## 3. Новые подписки и новые платящие

Канал берём у владельца подписки (`left join`, чтобы не потерять подписки, если владельца вдруг нет в `clients`).
Новый платящий — клиент, у которого первое в жизни успешное списание пришлось на этот месяц. Платежи сворачиваем до клиента **до** соединения с `clients`, чтобы не получить задвоения.

In [ ]:
new_subs = q("""
    select date_trunc('month', s.started_at)::date            as month,
           coalesce(c.acquisition_channel, :no_channel)       as channel,
           count(*)                                           as new_subs,
           count(*) filter (where s.is_trial)                 as new_trials
    from prime.subscriptions s
    left join prime.clients c on c.client_id = s.owner_client_id
    group by 1, 2
    order by 1, 2
""", no_channel=NO_CHANNEL)

total_subs = q("select count(*) as n from prime.subscriptions").iloc[0, 0]
assert new_subs["new_subs"].sum() == total_subs, "подписки по группам не сходятся с общим числом подписок"
print("подписок всего:", total_subs)

In [ ]:
new_payers = q("""
    with first_pay as (
        select s.owner_client_id, min(p.paid_at) as first_paid
        from prime.payments p
        join prime.subscriptions s on s.subscription_id = p.subscription_id
        where p.status = 'success'
        group by 1
    )
    select date_trunc('month', f.first_paid)::date            as month,
           coalesce(c.acquisition_channel, :no_channel)       as channel,
           count(*)                                           as new_payers
    from first_pay f
    left join prime.clients c on c.client_id = f.owner_client_id
    group by 1, 2
    order by 1, 2
""", no_channel=NO_CHANNEL)

total_payers = q("""
    select count(distinct s.owner_client_id) as n
    from prime.payments p
    join prime.subscriptions s on s.subscription_id = p.subscription_id
    where p.status = 'success'
""").iloc[0, 0]
assert new_payers["new_payers"].sum() == total_payers, "новые платящие не сходятся с числом платящих владельцев"
print("клиентов, которые платили хотя бы раз:", total_payers)

## 4. Сводная таблица «месяц × канал»

Три таблицы имеют одно зерно (месяц × канал), но месяцы у них не совпадают: платежи и подписки начинаются позже регистраций.
Поэтому склеиваем через `outer`, пропуски заменяем нулями (в этот месяц в этом канале события не было) и объявляем кратность `1:1`.

In [ ]:
KEYS = ["month", "channel"]

flow = (reg
        .merge(new_subs, on=KEYS, how="outer", validate="1:1")
        .merge(new_payers, on=KEYS, how="outer", validate="1:1")
        .fillna(0))

for col in ["registered", "new_subs", "new_trials", "new_payers"]:
    flow[col] = flow[col].astype(int)

flow["month"] = pd.to_datetime(flow["month"])
flow = flow.sort_values(KEYS).reset_index(drop=True)

# Сверка: ни одна группа не потерялась при склейке.
assert flow["registered"].sum() == reg["registered"].sum()
assert flow["new_subs"].sum() == new_subs["new_subs"].sum()
assert flow["new_payers"].sum() == new_payers["new_payers"].sum()
print("строк в таблице:", len(flow))
flow.tail(8)

In [ ]:
# Итог по месяцам (все каналы вместе) и приток в пересчёте на день: месяцы разной длины.
monthly = flow.groupby("month")[["registered", "new_subs", "new_trials", "new_payers"]].sum()
last_day = q("select greatest((select max(started_at) from prime.subscriptions), (select max(paid_at) from prime.payments)) as d").iloc[0, 0]

monthly["days"] = monthly.index.days_in_month
# В последнем месяце данные кончаются раньше его конца: делим на реально прошедшие дни.
last_month = monthly.index.max()
monthly.loc[last_month, "days"] = min(monthly.loc[last_month, "days"], last_day.day)
monthly["registered_per_day"] = (monthly["registered"] / monthly["days"]).round(1)
monthly["growth_per_day_%"] = (monthly["registered_per_day"].pct_change() * 100).round(1)
monthly

## 5. Конверсия регистрации в подписку (по когортам)

Здесь считаем иначе: берём **когорту** — клиентов, пришедших в месяце, — и смотрим, какая доля из них оформила подписку в течение 30 дней после регистрации.
Окно пишем рядом с конверсией всегда. Когорты, у которых с регистрации не прошло 30 дней, исключаем: им ещё не дали времени.

In [ ]:
WINDOW_DAYS = 30

# Крайний день данных: до него и считаем, чтобы окно у каждой когорты было полным.
cutoff = (last_day - pd.Timedelta(days=WINDOW_DAYS)).normalize()
print("последний день данных:", last_day, "| когорты берём до:", cutoff)

cohort = q("""
    with first_sub as (
        select owner_client_id, min(started_at) as first_start
        from prime.subscriptions
        group by 1
    )
    select date_trunc('month', c.registered_at)::date          as month,
           coalesce(c.acquisition_channel, :no_channel)        as channel,
           count(*)                                            as registered,
           count(*) filter (
               where f.first_start < c.registered_at + make_interval(days => :w)
           )                                                   as subscribed_in_window
    from prime.clients c
    left join first_sub f on f.owner_client_id = c.client_id
    where c.registered_at < :cutoff
    group by 1, 2
    order by 1, 2
""", no_channel=NO_CHANNEL, w=WINDOW_DAYS, cutoff=cutoff.to_pydatetime())

cohort["month"] = pd.to_datetime(cohort["month"])
by_month = cohort.groupby("month")[["registered", "subscribed_in_window"]].sum()
by_month[f"conversion_{WINDOW_DAYS}d_%"] = (by_month["subscribed_in_window"] / by_month["registered"] * 100).round(1)
by_month

In [ ]:
by_channel = cohort.groupby("channel")[["registered", "subscribed_in_window"]].sum()
by_channel[f"conversion_{WINDOW_DAYS}d_%"] = (by_channel["subscribed_in_window"] / by_channel["registered"] * 100).round(1)
by_channel.sort_values(f"conversion_{WINDOW_DAYS}d_%")

## 6. График: приток по месяцам

Приток на день, а не за месяц, чтобы длина месяца не создавала ложных скачков. Последний месяц, скорее всего, неполный и выделен штриховкой.
Один акцентный цвет, подписи осей, период в подписи.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.6), layout="constrained")

x = monthly.index
bars = ax.bar(x, monthly["registered_per_day"], width=20, color="#2A6F97", zorder=2)
bars[-1].set_hatch("//")
bars[-1].set_alpha(0.55)

ax.set_title("Новых клиентов в день по месяцам регистрации", loc="left")
ax.set_ylabel("клиентов в день")
ax.set_xlabel("Месяц регистрации (UTC). Штриховка: последний месяц данных, возможно неполный.")
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="y", color="#E5E7EB", zorder=0)
plt.show()

## 7. Что записать в записку после запуска

- Растёт ли приток регистраций? Есть ли месяцы с заметным скачком или провалом (в пересчёте на день)?
- Какие каналы дают основную долю притока и менялась ли эта доля?
- Отличается ли конверсия в подписку (за 30 дней) между каналами и когортами?
- Какие месяцы нельзя сравнивать из-за неполных данных.

Из ответов выбираем гипотезу и переходим к метрикам по каналам (CAC, отток, LTV) и факторной декомпозиции.